# 🔬 mmBERT (ModernBERT Architecture) 22-Layer Probing & Layer Selection
**Mục tiêu:** Điều tra định lượng toàn bộ 22 layers của `jhu-clsp/mmBERT-base` trên GPU T4 (Google Colab) nhằm chọn ra **4 layers tối ưu** cho bài toán **Cross-Lingual QA & Optimal Transport (OT)** trước khi bước vào pipeline huấn luyện Stage 1 và Stage 2.

### 📌 Đặc điểm kiến trúc ModernBERT cần lưu ý:
1. **Tổng cộng 22 layers** (chứ không phải 12 layer như BERT-base hay 24 layer như BERT-large).
2. **Alternating Attention:**
   * **Global Attention (Full Receptive Field):** Chu kỳ mỗi 3 layers (Layer index: `0, 3, 6, 9, 12, 15, 18, 21`).
   * **Local Sliding-Window Attention:** Các layer còn lại với window size chỉ 128 tokens.
   * *Hệ quả với QA:* Nếu dùng layer Local Attention, token ở cuối context (dài 384 tokens) sẽ không nhìn thấy câu hỏi, làm sai lệch ma trận chi phí OT!

---

In [ ]:
# ── [1] Cài đặt dependencies (yêu cầu transformers >= 4.48.0 để hỗ trợ ModernBERT) ──
!pip install -q "transformers>=4.48.0" peft accelerate datasets matplotlib seaborn pandas scipy

In [ ]:
# ── [2] Nạp Hugging Face Token & Đăng nhập ──
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✓ Đã tìm thấy HF_TOKEN trong Colab Secrets (Secrets icon 🔑 bên trái)!")
except Exception:
    pass

if not hf_token:
    import getpass
    print("Nếu bạn có Hugging Face Token, hãy dán vào bên dưới (hoặc nhấn Enter để bỏ qua):")
    hf_token = getpass.getpass("HF Token: ")

if hf_token and hf_token.strip():
    login(token=hf_token.strip())
    print("✓ Đã đăng nhập Hugging Face Hub thành công!")
else:
    print("ℹ Tiếp tục chế độ Public (tải trực tiếp mmBERT và XQuAD công khai).")

In [ ]:
# ── [3] Kiểm tra phần cứng GPU ──
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"  ✓ GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"  ✓ VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("  ⚠ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Runtime -> Change runtime type -> Chọn T4 GPU!")

In [ ]:
# ── [4] Thuật toán Sinkhorn OT & Hàm Nạp Dữ Liệu Song Ngữ XQuAD ──
import json
import random
import numpy as np
import pandas as pd
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

def sinkhorn_log_domain(C, en_pad_mask, tgt_pad_mask, epsilon=0.1, num_iters=50):
    """
    Batched Log-domain Sinkhorn-Knopp algorithm với non-uniform marginals.
    C: (B, M, N) cost matrix
    en_pad_mask: (B, M) True = PAD token
    tgt_pad_mask: (B, N) True = PAD token
    """
    B, M, N = C.shape
    valid_en = (~en_pad_mask).float()
    valid_tgt = (~tgt_pad_mask).float()
    n_en = valid_en.sum(dim=1, keepdim=True).clamp(min=1.0)
    n_tgt = valid_tgt.sum(dim=1, keepdim=True).clamp(min=1.0)

    log_mu = torch.where(en_pad_mask, torch.tensor(-1e9, device=C.device), -torch.log(n_en))
    log_nu = torch.where(tgt_pad_mask, torch.tensor(-1e9, device=C.device), -torch.log(n_tgt))

    u = torch.zeros(B, M, device=C.device)
    v = torch.zeros(B, N, device=C.device)
    log_K = -C / epsilon

    for _ in range(num_iters):
        u = log_mu - torch.logsumexp(log_K + v.unsqueeze(1), dim=2)
        v = log_nu - torch.logsumexp(log_K + u.unsqueeze(2), dim=1)

    log_gamma = log_K + u.unsqueeze(2) + v.unsqueeze(1)
    gamma = torch.exp(log_gamma)
    gamma = gamma.masked_fill(en_pad_mask.unsqueeze(2), 0.0)
    gamma = gamma.masked_fill(tgt_pad_mask.unsqueeze(1), 0.0)
    return gamma

def compute_transport_entropy(gamma, en_pad_mask, eps=1e-12):
    """
    Shannon entropy trung bình trên từng hàng (source token) của ma trận transport plan gamma.
    """
    row_sum = gamma.sum(dim=-1, keepdim=True).clamp(min=eps)
    p = gamma / row_sum
    h_tokens = -(p * (p + eps).log()).sum(dim=-1)
    valid_mask = (~en_pad_mask).float()
    return ((h_tokens * valid_mask).sum() / valid_mask.sum().clamp(min=1.0)).item()

def load_parallel_xquad(lang="vi", num_samples=120, seed=42):
    """
    Tải các cặp câu hỏi + context song ngữ (EN, Target) trực tiếp từ Hugging Face XQuAD.
    """
    ds_en = load_dataset("google/xquad", "xquad.en", split="validation")
    ds_tgt = load_dataset("google/xquad", f"xquad.{lang}", split="validation")

    en_map = {r["id"]: {"q": r["question"], "c": r["context"]} for r in ds_en}
    pairs = []
    for r in ds_tgt:
        qid = r["id"]
        if qid in en_map:
            pairs.append({
                "id": qid,
                "en_q": en_map[qid]["q"],
                "en_c": en_map[qid]["c"],
                "tgt_q": r["question"],
                "tgt_c": r["context"]
            })
    rng = random.Random(seed)
    rng.shuffle(pairs)
    return pairs[:num_samples]

print("✓ Đã sẵn sàng các hàm phụ trợ tính toán OT và tải dữ liệu.")

In [ ]:
# ── [5] Chạy Probing 22 Layers trên mmBERT (Vietnamese, Arabic, Hindi) ──
MODEL_NAME = "jhu-clsp/mmBERT-base"
LANGUAGES = ["vi", "ar", "hi"]
NUM_SAMPLES = 100    # 100 mẫu mỗi ngôn ngữ là đủ hội tụ thống kê
BATCH_SIZE = 8
MAX_LENGTH = 384

print(f"Đang tải Model & Tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = 0
if tokenizer.sep_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.sep_token_id = tokenizer.eos_token_id

model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(device)
model.eval()

num_layers = getattr(model.config, "num_hidden_layers", 22)
layer_types_cfg = getattr(model.config, "layer_types", None)
global_every_n = getattr(model.config, "global_attn_every_n_layers", 3)

layer_types = []
for l in range(num_layers):
    if layer_types_cfg is not None and l < len(layer_types_cfg):
        t = layer_types_cfg[l]
        layer_types.append("Global" if ("full" in t.lower() or "global" in t.lower()) else "Local")
    else:
        layer_types.append("Global" if (l % global_every_n == 0) else "Local")

print(f"Backbone có {num_layers} layers.")
globals_idx = [i for i, t in enumerate(layer_types) if t == "Global"]
locals_idx = [i for i, t in enumerate(layer_types) if t == "Local"]
print(f"  → Global Attention Layers ({len(globals_idx)}): {globals_idx}")
print(f"  → Local Attention Layers  ({len(locals_idx)}): {locals_idx}")

records = []

for lang in LANGUAGES:
    print(f"\n▶ Đang probe ngôn ngữ: {lang.upper()}...")
    pairs = load_parallel_xquad(lang=lang, num_samples=NUM_SAMPLES)
    
    layer_ot_sim = {i: [] for i in range(num_layers)}
    layer_entropy = {i: [] for i in range(num_layers)}

    for b in range(0, len(pairs), BATCH_SIZE):
        batch = pairs[b : b + BATCH_SIZE]
        en_texts = [f"{p['en_q']} {tokenizer.sep_token} {p['en_c']}" for p in batch]
        tgt_texts = [f"{p['tgt_q']} {tokenizer.sep_token} {p['tgt_c']}" for p in batch]

        enc_en = tokenizer(en_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        enc_tgt = tokenizer(tgt_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)

        with torch.no_grad():
            out_en = model(**enc_en)
            out_tgt = model(**enc_tgt)

        en_pad = (enc_en.attention_mask == 0)
        tgt_pad = (enc_tgt.attention_mask == 0)

        for l in range(num_layers):
            h_en = out_en.hidden_states[l + 1]
            h_tgt = out_tgt.hidden_states[l + 1]

            h_en_norm = F.normalize(h_en, p=2, dim=-1)
            h_tgt_norm = F.normalize(h_tgt, p=2, dim=-1)
            C = 1.0 - torch.bmm(h_en_norm, h_tgt_norm.transpose(1, 2))
            C = C.masked_fill(en_pad.unsqueeze(2), 1e4)
            C = C.masked_fill(tgt_pad.unsqueeze(1), 1e4)

            gamma = sinkhorn_log_domain(C, en_pad, tgt_pad, epsilon=0.1, num_iters=50)
            ot_cost = (gamma * C.masked_fill(C >= 1e3, 0.0)).sum(dim=(-2, -1)).mean().item()
            ot_sim = max(0.0, 1.0 - ot_cost)
            layer_ot_sim[l].append(ot_sim)

            ent = compute_transport_entropy(gamma, en_pad)
            layer_entropy[l].append(ent)

    for l in range(num_layers):
        records.append({
            "language": lang.upper(),
            "layer_index": l,
            "attention_type": layer_types[l],
            "ot_cosine_sim": float(np.mean(layer_ot_sim[l])),
            "transport_entropy": float(np.mean(layer_entropy[l]))
        })

df_results = pd.DataFrame(records)
df_results.to_csv("mmbert_layer_metrics.csv", index=False)
print("\n✓ Hoàn tất probing! Đã lưu kết quả vào file mmbert_layer_metrics.csv")

In [ ]:
# ── [6] Trực Quan Hóa & Khuyến Nghị 4 Layer Tối Ưu Cho Bài Báo ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
colors = {"VI": "#1f77b4", "AR": "#2ca02c", "HI": "#d62728"}

# Top: OT Alignment
ax1 = axes[0]
for lang in df_results["language"].unique():
    sub = df_results[df_results["language"] == lang].sort_values("layer_index")
    ax1.plot(sub["layer_index"], sub["ot_cosine_sim"], label=f"{lang} (OT-Cosine)", color=colors.get(lang, "gray"), linewidth=2)
    
    # Dấu chấm tròn cho Global, dấu x cho Local
    g_sub = sub[sub["attention_type"] == "Global"]
    ax1.scatter(g_sub["layer_index"], g_sub["ot_cosine_sim"], color=colors.get(lang, "gray"), s=70, marker="o", zorder=5)
    l_sub = sub[sub["attention_type"] == "Local"]
    ax1.scatter(l_sub["layer_index"], l_sub["ot_cosine_sim"], color=colors.get(lang, "gray"), s=35, marker="x", alpha=0.6, zorder=4)

ax1.set_ylabel("Cross-Lingual OT Alignment\n(1 - OT Cost)", fontsize=11, fontweight="bold")
ax1.set_title("mmBERT 22-Layer Probing: Representation Alignment & Transport Entropy", fontsize=13, fontweight="bold", pad=10)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="lower right")

# Shading candidate zone
ax1.axvspan(10.5, 21.5, color="gold", alpha=0.15, label="Candidate Zone (Mid-to-Deep)")

# Bottom: Transport Entropy
ax2 = axes[1]
for lang in df_results["language"].unique():
    sub = df_results[df_results["language"] == lang].sort_values("layer_index")
    ax2.plot(sub["layer_index"], sub["transport_entropy"], label=f"{lang} (Entropy)", color=colors.get(lang, "gray"), linewidth=2)
    g_sub = sub[sub["attention_type"] == "Global"]
    ax2.scatter(g_sub["layer_index"], g_sub["transport_entropy"], color=colors.get(lang, "gray"), s=70, marker="o", zorder=5)

ax2.axvspan(10.5, 21.5, color="gold", alpha=0.15)
ax2.set_xlabel("Layer Index (0 đến 21)", fontsize=11, fontweight="bold")
ax2.set_ylabel("Sinkhorn Entropy H(γ)\n(Thấp hơn = Càng sắc nét)", fontsize=11, fontweight="bold")
ax2.set_xticks(range(num_layers))
xtick_labels = [f"L{i}\n(G)" if layer_types[i] == "Global" else f"L{i}" for i in range(num_layers)]
ax2.set_xticklabels(xtick_labels, fontsize=9)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig("mmbert_layer_probing.png", dpi=300)
plt.show()

# Tính điểm trung bình và đưa ra khuyến nghị
avg_df = df_results.groupby(["layer_index", "attention_type"])[["ot_cosine_sim", "transport_entropy"]].mean().reset_index()
globals_cand = avg_df[(avg_df["attention_type"] == "Global") & (avg_df["layer_index"] >= 10)].sort_values("layer_index")
recommended_layers = globals_cand["layer_index"].tolist()[-4:]

print("=" * 70)
print(f"🌟 KHUYẾN NGHỊ TẬP TARGET LAYERS TỐI ƯU CHO mmBERT: {recommended_layers}")
print("Lý do khoa học:")
print("  1. 100% Global Attention: Bao phủ toàn diện receptive field 512 tokens cho cả câu hỏi và ngữ cảnh.")
print("  2. Tầng ngữ nghĩa giữa-sâu (50% - 95%): Cân bằng giữa khả năng đồng chỉnh liên ngôn ngữ và tính phân biệt của span boundaries.")
print("  3. Cờ dòng lệnh (CLI flag) để đưa vào huấn luyện Stage 1 và Stage 2:")
print(f"     --target_layers {' '.join(str(l) for l in recommended_layers)}")
print("=" * 70)

In [ ]:
# ── [7] Tải kết quả về máy tính cá nhân ──
from google.colab import files
print("Đang chuẩn bị tải về biểu đồ và file CSV số liệu...")
files.download("mmbert_layer_probing.png")
files.download("mmbert_layer_metrics.csv")
print("✓ Hoàn tất!")